# Otter: single-species electronic-to-ionic workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/otter-hed/otter/blob/main/notebooks/00-otter_intro.ipynb)

This notebook contains the complete calculation implemented in `examples/single_species_workflow.py`. It evaluates an average atom, constructs the pseudoatom and effective ion–ion potential, and solves QOZ/HNC for $g_{ii}(r)$ and $S_{ii}(k)$. The central method follows [Starrett and Saumon (2014)](https://doi.org/10.1016/j.hedp.2013.12.001).

## Install Otter

Use a fresh Colab runtime.

In [ ]:
%pip install -q git+https://github.com/otter-hed/otter.git

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import otter
from otter import PlasmaWorkflowConfig, solve_plasma_workflow
from otter.plotting import grid_figsize, set_style

print(f"Otter {otter.__version__}")

## Input

The default is aluminium at $\rho=8.1\,\mathrm{g\,cm^{-3}}$ and $T_e=T_i=15\,\mathrm{eV}$.

In [ ]:
ELEMENT = "Al"
RHO_G_CC = 8.1
TE_EV = 15.0
TI_EV = 15.0

## Calculate

The default quantum continuum calculation can take ~ minute on a Colab CPU.

In [ ]:
config = PlasmaWorkflowConfig(
    elements=[ELEMENT],
    temperature_ev=TE_EV,
    rho_g_cc=RHO_G_CC,
    ion_temperature_ev=TI_EV,
    show_progress=True,
)

result = solve_plasma_workflow(config)
ion = result["ion"]
electronic = result["electronic"]["result"]

## Numerical summary

In [ ]:
print(f"element={ELEMENT} Te={TE_EV:g} eV Ti={TI_EV:g} eV rho={RHO_G_CC:g} g/cc")
print(f"mu={electronic['mu']:.8f} Ha  zbar={electronic['zbar']:.8f}")
print(
    "threshold="
    f"{electronic.get('threshold_state_status', 'none')}  "
    f"representation={electronic.get('threshold_state_representation', 'none')}  "
    f"E_shallow={float(electronic.get('shallowest_bound_energy_ha', np.nan)):.8e} Ha"
)
print(
    f"HNC iterations={ion['hnc_iters']}  "
    f"qoz={ion['qoz_build_s']:.3f}s hnc={ion['hnc_solve_s']:.3f}s"
)

## Plotting

In [ ]:
set_style("docs", palette="deep_science")
fig, axes = plt.subplots(
    2, 2, figsize=grid_figsize(2, 2), constrained_layout=True
)
ax_density, ax_potential, ax_gii, ax_sii = axes.ravel()

r_e = np.asarray(electronic["r"], dtype=float)
density_fields = {
    "n_full": r"$n_{\mathrm{full}}$",
    "n_bound": r"$n_{\mathrm{bound}}$",
    "n_cont": r"$n_{\mathrm{cont}}$",
    "n_ion": r"$n_{\mathrm{ion}}$",
    "n_ext": r"$n_{\mathrm{ext}}$",
    "n_pa": r"$n_{\mathrm{pa}}$",
    "n_scr": r"$n_{\mathrm{scr}}$",
}
for name, label in density_fields.items():
    if name in electronic:
        values = np.asarray(electronic[name], dtype=float)
        if values.shape == r_e.shape:
            ax_density.plot(
                r_e,
                4.0 * np.pi * r_e**2 * values,
                label=label,
            )
if "n0" in electronic:
    n0_profile = np.full_like(r_e, float(electronic["n0"]))
    ax_density.plot(
        r_e,
        4.0 * np.pi * r_e**2 * n0_profile,
        color="black",
        linestyle="--",
        linewidth=1.3,
        label=r"$ n_0$",
    )
r_ws = electronic.get("r_ws") or electronic.get("meta", {}).get("r_ws_bohr")
if r_ws is not None:
    r_ws = float(r_ws)
if r_ws is not None:
    ax_density.axvline(r_ws, color="black", linestyle=":", linewidth=1.1, label=r"$R_{\mathrm{ws}}$")
ax_density.set_xlabel(r"$r\,[a_0]$")
ax_density.set_ylabel(r"$4\pi r^2 n(r)\,[a_0^{-1}]$")
ax_density.set_ylim(-0.5, 15)
ax_density.set_xlim(-0.5, 8)
ax_density.legend()

if "v_full" in electronic:
    ax_potential.plot(r_e, np.asarray(electronic["v_full"]), label=r"$V_{\mathrm{eff}}^{\mathrm{full}}$")
if "v_ext" in electronic:
    ax_potential.plot(r_e, np.asarray(electronic["v_ext"]), label=r"$V_{\mathrm{eff}}^{\mathrm{ext}}$")
ax_potential.axhline(0.0, color="gray", linestyle="--", linewidth=1.0)
if r_ws is not None:
    ax_potential.axvline(r_ws, color="black", linestyle=":", linewidth=1.1, label=r"$R_{\mathrm{ws}}$")
ax_potential.set_xlabel(r"$r\,[a_0]$")
ax_potential.set_ylabel(r"$V_{\mathrm{eff}}(r)\,[\mathrm{Ha}]$")
ax_potential.set_xlim(-0.5, 8.0)
ax_potential.set_ylim(-1.0, 1.0)
ax_potential.legend()

r_ion = np.asarray(ion["r"], dtype=float)
ax_gii.plot(r_ion, ion["gii_r"], label=r"$g_{ii}(r)$")
ax_gii.set_xlabel(r"$r\,[a_0]$")
ax_gii.set_ylabel(r"$g_{ii}(r)$")
ax_gii.set_xlim(-0.5, 20.0)
ax_gii.legend()

k_ion = np.asarray(ion["k"], dtype=float)
ax_sii.plot(k_ion, ion["sii_k"], label=r"$S_{ii}(k)$")
ax_sii.set_xlabel(r"$k\,[a_0^{-1}]$")
ax_sii.set_ylabel(r"$S_{ii}(k)$")
ax_sii.set_xlim(0.0, 20.0)
ax_sii.legend()

plt.show()